# An Interpreter for a Simple Programming Language: Exercise Frame

This notebook is the starting point for the exercise at the end of the chapter *Implementing a Simple
Interpreter*.  It contains the interpreter developed in the notebook `03-Interpreter.ipynb`, extended by
*floating point numbers*, *strings*, and a number of *predefined functions*.  Your task is to extend this
interpreter as follows:

1. Add `for` loops to the interpreter.
2. Add the logical operators `&&` (*and*), `||` (*or*), and `!` (*not*).  The operator `!` should bind
   strongest and the operator `||` should bind weakest.
3. Add user-defined functions:
   - A function should only have access to its parameters and to those variables that are defined
     locally inside the function.
   - A function should always return a value using a `return` statement.  As a `return` statement can
     occur anywhere in a function, use an *exception* to communicate the return value and to transfer
     the control flow out of the function.

The places where you have to add code are marked with the comment `your code here`.  The notebook runs
as it is; only the example programs at the end need your extensions.  In OCaml, you also have to extend the types
of the abstract syntax trees, and the compiler will then tell you which functions do not yet handle the new cases.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

The external helper file `AST2Dot.ipynb` is used to represent our abstract syntax trees as graphs.  It draws
nested tuples of the type `nested`.

In [ ]:
run_notebook "AST2Dot.ipynb"

## Specification of the Grammar

Compared to the grammar of `03-Interpreter.ipynb`, the grammar below has the following additions:
- Assignments are described by the syntactical variable `assign`, so that they can also be used inside
  the head of a `for` loop.
- The tokens `FLOAT` and `STRING` describe floating point numbers and string literals.

Add the grammar rules for `for` loops, the logical operators, function definitions, and `return`
statements.  Remember to name the nodes of the parse tree in the semantic actions: then `ast_to_stmnt` only needs
new cases for these names.  For the logical operators, introduce the syntactical variables
`disjunction`, `conjunction`, and `negation` in order to encode the precedences.  Do not forget to add the new
tokens to the scanner specification and to the `%token` declarations of the grammar.

In [ ]:
let sl_lexer = {|
{
  open Sl_parser
}

let digits = ['0'-'9'] ('_'? ['0'-'9'])*
let float  = digits '.' digits (['e' 'E'] ['+' '-']? ['0'-'9']+)?
let string = '"' ('\\' _ | [^ '"' '\\'])* '"'
let cname  = ['_' 'a'-'z' 'A'-'Z'] ['_' 'a'-'z' 'A'-'Z' '0'-'9']*

rule token = parse
  | [' ' '\t' '\r' '\012']+ { token lexbuf }                     (* ignore whitespace *)
  | '\n'                    { Lexing.new_line lexbuf; token lexbuf }
  | "//" [^ '\n']*          { token lexbuf }                     (* CPP_COMMENT *)
  | "/*"                    { comment lexbuf; token lexbuf }     (* C_COMMENT *)
  | "if"                    { IF }
  | "while"                 { WHILE }
  (* your code here: the keywords of for loops, function definitions, and return statements *)
  | ":="                    { ASSIGN }
  | "=="                    { EQ }
  | "!="                    { NE }
  | "<="                    { LE }
  | ">="                    { GE }
  | '<'                     { LT }
  | '>'                     { GT }
  (* your code here: the logical operators ||, &&, and ! *)
  | '+'                     { PLUS }
  | '-'                     { MINUS }
  | '*'                     { STAR }
  | '/'                     { SLASH }
  | '%'                     { PERCENT }
  | '('                     { LPAR }
  | ')'                     { RPAR }
  | '{'                     { LBRACE }
  | '}'                     { RBRACE }
  | ';'                     { SEMICOLON }
  | ','                     { COMMA }
  | float                   { FLOAT (Parse_tree.token "FLOAT" lexbuf) }
  | digits                  { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
  | string                  { STRING (Parse_tree.token "STRING" lexbuf) }
  | cname                   { IDENTIFIER (Parse_tree.token "IDENTIFIER" lexbuf) }
  | eof                     { EOF }
  | _                       { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ Lexing.lexeme lexbuf)) }

and comment = parse
  | "*/"                    { () }
  | '\n'                    { Lexing.new_line lexbuf; comment lexbuf }
  | eof                     { raise (Parse_tree.unexpected lexbuf "unterminated comment") }
  | _                       { comment lexbuf }
|}

In [ ]:
let sl_grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> IDENTIFIER NUMBER FLOAT STRING
%token IF WHILE ASSIGN EQ NE LE GE LT GT PLUS MINUS STAR SLASH PERCENT
%token LPAR RPAR LBRACE RBRACE SEMICOLON COMMA EOF
/* your code here: the new tokens */

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> program stmnt assign bool_expr expr product factor expr_list
%type  <Parse_tree.tree list> stmnt_list expr_seq

%%

start: p = program EOF                                 { p }

program: l = stmnt_list                                { Tree ("prog", l) }

stmnt_list: /* empty */                                { [] }
          | s = stmnt l = stmnt_list                   { s :: l }

stmnt: IF LPAR b = bool_expr RPAR s = stmnt            { Tree ("if_stmnt", [b; s]) }
     | WHILE LPAR b = bool_expr RPAR s = stmnt         { Tree ("while_stmnt", [b; s]) }
     /* your code here: for loops */
     /* your code here: function definitions and return statements */
     | LBRACE p = program RBRACE                       { Tree ("block", [p]) }
     | a = assign SEMICOLON                            { a }
     | e = expr SEMICOLON                              { Tree ("expr_stmnt", [e]) }

assign: id = IDENTIFIER ASSIGN e = expr                { Tree ("assign", [Token id; e]) }

/* your code here: the logical operators ||, &&, and ! */
bool_expr: l = expr EQ r = expr                        { Tree ("eq", [l; r]) }
         | l = expr NE r = expr                        { Tree ("ne", [l; r]) }
         | l = expr LE r = expr                        { Tree ("le", [l; r]) }
         | l = expr GE r = expr                        { Tree ("ge", [l; r]) }
         | l = expr LT r = expr                        { Tree ("lt", [l; r]) }
         | l = expr GT r = expr                        { Tree ("gt", [l; r]) }

expr: e = expr PLUS p = product                        { Tree ("add", [e; p]) }
    | e = expr MINUS p = product                       { Tree ("sub", [e; p]) }
    | p = product                                      { p }

product: p = product STAR f = factor                   { Tree ("mul", [p; f]) }
       | p = product SLASH f = factor                  { Tree ("div", [p; f]) }
       | p = product PERCENT f = factor                { Tree ("mod", [p; f]) }
       | f = factor                                    { f }

factor: LPAR e = expr RPAR                             { e }
      | n = NUMBER                                     { Tree ("number", [Token n]) }
      | x = FLOAT                                      { Tree ("float_number", [Token x]) }
      | s = STRING                                     { Tree ("string", [Token s]) }
      | id = IDENTIFIER                                { Tree ("var", [Token id]) }
      | id = IDENTIFIER LPAR l = expr_list RPAR        { Tree ("fct_call", [Token id; l]) }

expr_list: /* empty */                                 { Tree ("explist", []) }
         | l = expr_seq                                { Tree ("explist", l) }

expr_seq: e = expr                                     { [e] }
        | e = expr COMMA l = expr_seq                  { e :: l }
|}

We generate the scanner and the parser.  `menhir --lalr` creates a *Look-Ahead Left-to-Right* (LALR) parser.

In [ ]:
generate_parser ~name:"sl" ~lexer:sl_lexer ~grammar:sl_grammar

## Generating the Abstract Syntax Tree

Our abstract syntax trees are values of the variant types `expr`, `bool_expr`, and `stmnt`.  Numbers are represented
as `Num n` or `FloatNum x`, variables as `Var v`, and string literals as `StrLit s`.

**Type safety:** In the Python version, both variables and string literals are strings, so a string literal is
represented by the string *including* its double quotes in order to distinguish it from a variable.  In OCaml, string
literals and variables are different constructors, so the quotes are removed.

The values of the language are integers, floating point numbers, and strings.  In Python, these are the types `int`,
`float`, and `str`.  In OCaml, they are the constructors `I`, `F`, and `S` of the type `value`.

In [ ]:
type arith_op = Add | Sub | Mul | Div | Mod
type cmp_op   = Eq | Ne | Le | Ge | Lt | Gt

type expr =
  | Num      of int
  | FloatNum of float
  | StrLit   of string
  | Var      of string
  | Call     of string * expr list        (* ('call', f_name, arg1, ..., argn) *)
  | Arith    of arith_op * expr * expr

type bool_expr =
  | Cmp of cmp_op * expr * expr
  (* your code here: the logical operators *)

type stmnt =
  | Prog   of stmnt list                  (* ('.', stmnt1, ..., stmntn)  *)
  | Assign of string * expr               (* (':=', var, expr)           *)
  | Expr   of expr                        (* ('expr', expr)              *)
  | If     of bool_expr * stmnt           (* ('if', test, stmnt)         *)
  | While  of bool_expr * stmnt           (* ('while', test, stmnt)      *)
  (* your code here: for loops, function definitions, and return statements *)

type value = I of int | F of float | S of string

`arith_map` and `cmp_map` map the names of the nodes of the parse tree to the operators.  They correspond to the
dictionary `SYMBOL_MAP` of the Python version.  The functions `arith_symbol` and `cmp_symbol` return the symbols
that are used for the operators in the Python version.

In [ ]:
let arith_map : (string * arith_op) list = [ ("add", Add); ("sub", Sub); ("mul", Mul); ("div", Div); ("mod", Mod) ]
let cmp_map   : (string * cmp_op) list   = [ ("eq", Eq); ("ne", Ne); ("le", Le); ("ge", Ge); ("lt", Lt); ("gt", Gt) ]
(* your code here: the logical operators *)

let arith_symbol (op : arith_op) : string =
  match op with Add -> "+" | Sub -> "-" | Mul -> "*" | Div -> "/" | Mod -> "%"

let cmp_symbol (op : cmp_op) : string =
  match op with Eq -> "==" | Ne -> "!=" | Le -> "<=" | Ge -> ">=" | Lt -> "<" | Gt -> ">"

`ast_to_stmnt` takes a node of the parse tree and transforms it into a statement.  It uses the functions `ast_to_expr`
and `ast_to_bool`, which transform nodes into expressions and tests.

In [ ]:
let rec ast_to_expr (node : tree) : expr =
  match node with
  | Tree ("number", [ Token t ]) -> Num (int_of_string t.value)
  | Tree ("float_number", [ Token t ]) -> FloatNum (float_of_string t.value)
  | Tree ("string", [ Token t ]) -> StrLit (String.sub t.value 1 (String.length t.value - 2))
  | Tree ("var", [ Token t ]) -> Var t.value
  | Tree ("fct_call", [ Token f; Tree ("explist", args) ]) -> Call (f.value, List.map ast_to_expr args)
  | Tree (name, [ lhs; rhs ]) when List.mem_assoc name arith_map ->
      Arith (List.assoc name arith_map, ast_to_expr lhs, ast_to_expr rhs)
  | _ -> invalid_arg ("Unknown syntax tree node: " ^ Parse_tree.to_string node)

let ast_to_bool (node : tree) : bool_expr =
  match node with
  | Tree (name, [ lhs; rhs ]) when List.mem_assoc name cmp_map ->
      Cmp (List.assoc name cmp_map, ast_to_expr lhs, ast_to_expr rhs)
  (* your code here: the logical operators *)
  | _ -> invalid_arg ("Unknown syntax tree node: " ^ Parse_tree.to_string node)

let rec ast_to_stmnt (node : tree) : stmnt =
  match node with
  | Tree ("prog", stmnts) -> Prog (List.map ast_to_stmnt stmnts)
  | Tree ("block", [ program ]) -> ast_to_stmnt program
  | Tree ("assign", [ Token var; e ]) -> Assign (var.value, ast_to_expr e)
  | Tree ("expr_stmnt", [ e ]) -> Expr (ast_to_expr e)
  | Tree ("if_stmnt", [ test; s ]) -> If (ast_to_bool test, ast_to_stmnt s)
  | Tree ("while_stmnt", [ test; s ]) -> While (ast_to_bool test, ast_to_stmnt s)
  (* your code here: for loops, function definitions with their lists of parameters, and return statements *)
  | _ -> invalid_arg ("Unknown syntax tree node: " ^ Parse_tree.to_string node)

The functions below convert an abstract syntax tree into a nested tuple that has the same structure as the nested
tuples of the Python version, so that it can be printed and drawn.  The function `parse` takes a `file_name` as its
sole argument.  The file is read, parsed, and transformed into an abstract syntax tree, which is then displayed as a graph.

In [ ]:
let rec expr_to_nested (e : expr) : nested =
  match e with
  | Num n -> Int n
  | FloatNum x -> Float x
  | StrLit s -> Str ("\"" ^ s ^ "\"")
  | Var v -> Str v
  | Call (f, args) -> Tuple ("call", Str f :: List.map expr_to_nested args)
  | Arith (op, lhs, rhs) -> Tuple (arith_symbol op, [ expr_to_nested lhs; expr_to_nested rhs ])

let bool_to_nested (test : bool_expr) : nested =
  match test with
  | Cmp (op, lhs, rhs) -> Tuple (cmp_symbol op, [ expr_to_nested lhs; expr_to_nested rhs ])

let rec to_nested (s : stmnt) : nested =
  match s with
  | Prog sl -> Tuple (".", List.map to_nested sl)
  | Assign (var, e) -> Tuple (":=", [ Str var; expr_to_nested e ])
  | Expr e -> Tuple ("expr", [ expr_to_nested e ])
  | If (test, s) -> Tuple ("if", [ bool_to_nested test; to_nested s ])
  | While (test, s) -> Tuple ("while", [ bool_to_nested test; to_nested s ])

let parse (file_name : string) : Graphviz.t =
  let program = read_file file_name in
  print_endline program;
  let tree = parse_string Sl_parser.start Sl_lexer.token program in
  let ast  = ast_to_stmnt tree in
  print_endline (nested_to_string (to_nested ast));
  tuple2dot (to_nested ast)

In [ ]:
Graphviz.display (parse "sum.sl")

## The Runtime System

A `return` statement can occur anywhere inside a function.  Therefore, it is implemented by raising the exception
`Return_value`, which carries the value that is returned.  The call of the function catches this exception.

In [ ]:
exception Return_value of value

The hash table `functions` stores the user-defined functions.  It maps the name of a function to a pair
consisting of the list of its parameters and its body.

In [ ]:
let functions : (string, string list * stmnt) Hashtbl.t = Hashtbl.create 16

The function `execute_tuple` executes a list of statements.  The hash table `values` maps the names of
variables to their values.  Since the functions `execute_tuple`, `execute`, `evaluate_bool`, `evaluate`, and
`call_builtin` call each other, the next cell creates forward references for them.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

type env = (string, value) Hashtbl.t

let execute_ref       : (stmnt -> env -> unit) ref               = ref not_yet_defined
let evaluate_bool_ref : (bool_expr -> env -> bool) ref           = ref not_yet_defined
let evaluate_ref      : (expr -> env -> value) ref               = ref not_yet_defined
let call_builtin_ref  : (string -> value list -> value) ref      = ref not_yet_defined

let execute_tuple (statementList : stmnt list) (values : env) : unit =
  List.iter (fun stmnt -> !execute_ref stmnt values) statementList

The function `execute` matches recursively against the shape of the statement.  The `while` loop of Python is
replaced by a recursive function `loop`.

In [ ]:
let rec execute (stmnt : stmnt) (values : env) : unit =
  match stmnt with
  | Prog sl -> execute_tuple sl values
  | Assign (var, value) -> Hashtbl.replace values var (!evaluate_ref value values)
  | Expr expr -> ignore (!evaluate_ref expr values)
  | If (test, stmnt) -> if !evaluate_bool_ref test values then execute stmnt values
  | While (test, stmnt) ->
      let rec loop () = if !evaluate_bool_ref test values then (execute stmnt values; loop ()) in
      loop ()
  (* your code here: for loops *)
  (* your code here: function definitions *)
  (* your code here: return statements (raise a Return_value exception) *)
;;
execute_ref := execute

The function `evaluate_bool` evaluates the test of a conditional or of a loop and returns a Boolean
value.  The function `compare_values` compares two values like Python does: numbers are compared by their values,
strings are compared lexicographically, and a number is never equal to a string.

In [ ]:
let to_float (v : value) : float =
  match v with
  | I n -> float_of_int n
  | F x -> x
  | S _ -> failwith "TypeError: a number is required"

let compare_values (op : cmp_op) (l : value) (r : value) : bool =
  match l, r with
  | S a, S b -> (
      match op with Eq -> a = b | Ne -> a <> b | Le -> a <= b | Ge -> a >= b | Lt -> a < b | Gt -> a > b)
  | S _, _ | _, S _ -> (
      match op with Eq -> false | Ne -> true | _ -> failwith "TypeError: strings and numbers can not be compared")
  | _ -> (
      let a, b = to_float l, to_float r in
      match op with Eq -> a = b | Ne -> a <> b | Le -> a <= b | Ge -> a >= b | Lt -> a < b | Gt -> a > b)

let evaluate_bool (expr : bool_expr) (values : env) : bool =
  match expr with
  | Cmp (op, lhs, rhs) -> compare_values op (!evaluate_ref lhs values) (!evaluate_ref rhs values)
  (* your code here: the logical operators &&, ||, and ! *)
;;
evaluate_bool_ref := evaluate_bool

The function `evaluate` computes the value of an expression.  A call of a function is either a call of a
user-defined function or a call of a predefined function.  The arithmetic follows Python: `+` concatenates
strings, `/` always returns a floating point number, and `%` returns a result that has the same sign as its second
argument.  The function `arith` applies an operator to two numbers: the result is an integer if both numbers are
integers.

In [ ]:
let arith (int_op : int -> int -> int) (float_op : float -> float -> float) (a : value) (b : value) : value =
  match a, b with
  | I x, I y -> I (int_op x y)
  | _ -> F (float_op (to_float a) (to_float b))

let python_mod (x : int) (y : int) : int = ((x mod y) + y) mod y
let python_fmod (x : float) (y : float) : float = x -. y *. Float.floor (x /. y)

let rec evaluate (expr : expr) (values : env) : value =
  match expr with
  | Num n -> I n
  | FloatNum x -> F x
  | StrLit s -> S s
  | Var var -> Hashtbl.find values var
  | Arith (op, lhs, rhs) -> (
      let l = evaluate lhs values in
      let r = evaluate rhs values in
      match op, l, r with
      | Add, S a, S b -> S (a ^ b)
      | Add, _, _ -> arith ( + ) ( +. ) l r
      | Sub, _, _ -> arith ( - ) ( -. ) l r
      | Mul, _, _ -> arith ( * ) ( *. ) l r
      | Div, _, _ -> F (to_float l /. to_float r)
      | Mod, _, _ -> arith python_mod python_fmod l r)
  | Call (f_name, _args) when Hashtbl.mem functions f_name ->
      (* your code here: call a user-defined function
          - evaluate the arguments,
          - execute the body with a new hash table that contains only the parameters,
          - the body returns its value by raising a Return_value exception. *)
      failwith ("your code here: call of " ^ f_name)
  | Call (f_name, args) -> !call_builtin_ref f_name (List.map (fun arg -> evaluate arg values) args)
;;
evaluate_ref := evaluate

The function `call_builtin` implements the predefined functions of our language.  The function `print`
prints all of its arguments without separators between them.  Numbers are printed like Python prints them.
The function `read` uses the function `input` from the file `../lib/nb.ml`, since the OCaml kernel can not read
from the keyboard.

In [ ]:
let value_to_string (v : value) : string =
  match v with
  | I n -> string_of_int n
  | F x -> float_repr x
  | S s -> s

let call_builtin (f_name : string) (args : value list) : value =
  match f_name, args with
  | "read", [] ->
      let s = input "Please enter a number: " in
      if String.contains s '.' || String.contains s 'e' || String.contains s 'E' then F (float_of_string s)
      else I (int_of_string s)
  | "print", _ ->
      print_endline (String.concat "" (List.map value_to_string args));
      I 0
  | "sqrt", [ x ] -> F (Float.sqrt (to_float x))
  | "exp", [ x ] -> F (Float.exp (to_float x))
  | "ln", [ x ] -> F (Float.log (to_float x))
  | "sin", [ x ] -> F (Float.sin (to_float x))
  | "cos", [ x ] -> F (Float.cos (to_float x))
  | "tan", [ x ] -> F (Float.tan (to_float x))
  | "arctan", [ x ] -> F (Float.atan (to_float x))
  | _ -> failwith (Printf.sprintf "function %s is unknown" f_name)
;;
call_builtin_ref := call_builtin

## Putting Everything Together

The function `main` reads a program from a file, parses it, and executes it.  If `show_ast` is `true`, the
abstract syntax tree is displayed first.  `show_ast` is an optional labelled argument, which is passed as
`~show_ast:true`.

In [ ]:
let main ?(show_ast : bool = false) (file_name : string) : unit =
  let program = read_file file_name in
  let tree = parse_string Sl_parser.start Sl_lexer.token program in
  let ast  = ast_to_stmnt tree in
  if show_ast then Graphviz.display (tuple2dot (to_nested ast));
  Hashtbl.reset functions;
  execute ast (Hashtbl.create 16)

The program `sum.sl` only uses features that are already implemented.  It calls `read()`, so we first tell the
function `input` which number the user enters.

In [ ]:
cat "sum.sl"

In [ ]:
simulate_input [ "10" ];;
main "sum.sl"

## Testing Your Extensions

The following programs need your extensions.  The program `factorial.sl` uses user-defined functions and
a `for` loop.  As long as the extensions are missing, the parser reports a syntax error for these programs.

In [ ]:
cat "Exercise/factorial.sl"

In [ ]:
main "Exercise/factorial.sl"

In [ ]:
cat "Exercise/e.sl"

In [ ]:
main "Exercise/e.sl"

In [ ]:
cat "Exercise/pi.sl"

In [ ]:
main "Exercise/pi.sl"

In [ ]:
cat "Exercise/solve.sl"

In [ ]:
main "Exercise/solve.sl"

In [ ]:
cat "Exercise/sum-for.sl"

In [ ]:
simulate_input [ "10" ];;
main "Exercise/sum-for.sl"